# Asistente de compras por voz

Convierte voz a texto, permite corregir la transcripción, consulta productos y la tienda, y responde con audio sintetizado por XTTS. El procesamiento corre en el runtime local.

In [3]:
%pip install "transformers==4.46.2"

Note: you may need to restart the kernel to use updated packages.


In [4]:
import transformers
from transformers import BeamSearchScorer

print(transformers.__version__)

4.46.2


In [5]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "src" / "lab4_asistente").is_dir():
    ROOT = next((p for p in ROOT.parents if (p / "src" / "lab4_asistente").is_dir()), ROOT)
sys.path.insert(0, str(ROOT / "src"))
print("Raíz del proyecto:", ROOT)
from pathlib import Path
from IPython.display import Audio, display
from lab4_asistente import config as C
from lab4_asistente.runtime import cargar_sistema, cargar_modelos_voz


Raíz del proyecto: /home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz


## Cargar el sistema y medir cargas iniciales

In [ ]:
COMP = cargar_sistema(cargar_llm=True)
SIS = COMP["sistema"]
VOICE_ID = "winz"  # cambia al nombre del WAV y del perfil .pt elegido
ASR, TTS, cargas_voz = cargar_modelos_voz(VOICE_ID, COMP["dispositivo"])
CARGAS_INICIALES = {**COMP["cargas"], **cargas_voz}
print("Cargas iniciales (fuera de la latencia por turno):", CARGAS_INICIALES)


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

 > tts_models/multilingual/multi-dataset/xtts_v2 is already downloaded.
 > Using model: xtts


/home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz/.venv-dl-cuda/lib/python3.11/site-packages/TTS/tts/layers/xtts/xtts_manager.py:5: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  self.speakers = torch.load(speaker_file_path)
/home/winzcode/proyectos/ADEEP_LEARNING/asistente-compras-por-voz/.venv-dl-cuda/lib/python3.11/site-packages/TTS/utils/io.py:54: UserWarning: Environment variable TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD detected, since the`weights_only` argument was not explicitly passed to `torch.load`, forcing weights_only=False.
  return torch.load(f, map_location=map_location, **kwargs)


## Procesar una consulta hablada y permitir corregirla

In [ ]:
from google.colab import files

subidos = files.upload()  # selecciona un WAV en español
ruta_audio = Path(next(iter(subidos)))
transcripcion = ASR.transcribir(ruta_audio)
print("Transcripción ASR:", transcripcion)
corregida = input("Corrige la transcripción si hace falta; Enter conserva la original: ").strip()
entrada = corregida or transcripcion

respuesta = SIS.agente.turno(entrada)
print("Respuesta:", respuesta["respuesta"])
print("Operaciones:", [(o["herramienta"], o["resultado"].get("ok")) for o in respuesta["operaciones"]])
ruta_salida = C.RUTA_OUTPUT / "respuesta_demo.wav"
ruta_salida.parent.mkdir(parents=True, exist_ok=True)
TTS.sintetizar(respuesta["respuesta"], ruta_salida)
display(Audio(filename=str(ruta_salida)))


## Flujo de la consulta

La transcripción se revisa antes de enviarla al agente. El agente consulta el catálogo y la tienda mediante herramientas, y pide confirmación después de mostrar la cotización. XTTS usa `data/voice_profiles/<VOICE_ID>.pt`.

## Batería de pruebas de la consulta por voz

Ejecuta estas comprobaciones después de procesar una grabación. Verás qué texto llegó al agente, qué respondió, qué operaciones intentó y si se generó el audio de salida.

In [ ]:
import pandas as pd

entrada_usada = globals().get("entrada", "")
transcripcion_actual = globals().get("transcripcion", "")
respuesta_actual = globals().get("respuesta", {})
operaciones_actuales = respuesta_actual.get("operaciones", []) if isinstance(respuesta_actual, dict) else []
ruta_wav_actual = globals().get("ruta_salida", C.RUTA_OUTPUT / "respuesta_demo.wav")

pruebas_consulta_voz = [
    {"Prueba": "1. Transcripción ASR", "Esperado": "La grabación produce texto",
     "Resultado real": str(transcripcion_actual), "ok": bool(str(transcripcion_actual).strip())},
    {"Prueba": "2. Texto enviado al agente", "Esperado": "Se envía la transcripción o su corrección",
     "Resultado real": str(entrada_usada), "ok": bool(str(entrada_usada).strip())},
    {"Prueba": "3. Respuesta del asistente", "Esperado": "El agente devuelve una respuesta no vacía",
     "Resultado real": str(respuesta_actual.get("respuesta", "")) if isinstance(respuesta_actual, dict) else str(respuesta_actual),
     "ok": bool(respuesta_actual.get("respuesta", "").strip()) if isinstance(respuesta_actual, dict) else bool(str(respuesta_actual).strip())},
    {"Prueba": "4. Registro de operaciones", "Esperado": "Las operaciones del agente quedan visibles",
     "Resultado real": str([(o.get("herramienta"), o.get("resultado", {}).get("ok")) for o in operaciones_actuales]),
     "ok": isinstance(respuesta_actual, dict) and "operaciones" in respuesta_actual},
    {"Prueba": "5. Audio sintetizado", "Esperado": "Existe respuesta_demo.wav con contenido",
     "Resultado real": str(ruta_wav_actual),
     "ok": Path(ruta_wav_actual).is_file() and Path(ruta_wav_actual).stat().st_size > 44},
]
tabla_pruebas_voz = pd.DataFrame([{k: v for k, v in p.items() if k != "ok"} |
                                    {"Estado": "OK" if p["ok"] else "REVISAR"}
                                   for p in pruebas_consulta_voz])
display(tabla_pruebas_voz)